# Yotuubef Colab Notebook (All Features, Fresh Build)

This notebook runs the current project on Google Colab with full CLI command coverage and Google Drive persistence.

What this notebook handles:
- Uses a Google Drive folder for background videos (`BACKGROUND_FOLDER`).
- Persists important artifacts to Drive (`findings`, `processed`, `data/results`, `logs`, database files, token file).
- Lets you run every command from `main.py`: `find`, `single`, `batch`, `hybrid`, `manage`, `optimize`, `status`, `cleanup`.

Before running:
1. In Colab, set **Runtime -> Change runtime type -> GPU**.
2. Add Colab Secrets (left sidebar key icon):
   - Required for core generation: `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET`, `NVIDIA_NIM_API_KEY`
   - Recommended/optional: `REDDIT_USER_AGENT`, `NVIDIA_NIM_MODEL`, `NVIDIA_NIM_ALT_MODEL`, `NVIDIA_NIM_BASE_URL`, `GEMINI_API_KEY`, `BRAVE_SEARCH_API_KEY`, `HACKCLUB_SEARCH_API_KEY`
   - Optional for YouTube upload auth: `YOUTUBE_TOKEN_JSON`
3. Put one or more `.mp4` files in the Drive backgrounds folder shown by the setup cells.
4. **YouTube video downloads (hybrid workflow):** If downloads fail with 403, set `YTDLP_COOKIES_FROM_BROWSER` (e.g. `chrome`) when running locally with a logged-in browser, or use `YTDLP_COOKIES_FILE` pointing to a Netscape-format cookies file.


In [ ]:
# Cell 1: System + repo + dependencies
import os
import shlex
import subprocess
from pathlib import Path

def run(cmd: str, allow_fail: bool = False) -> None:
    print(f"$ {cmd}")
    completed = subprocess.run(cmd, shell=True, text=True)
    if completed.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")

run("nvidia-smi", allow_fail=True)
run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg sox git gcc g++")

REPO_URL = "https://github.com/beenycool/yotuubef.git"
REPO_DIR = Path("/content/yotuubef")

if not REPO_DIR.exists():
    run(f"git clone {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}")

os.chdir(REPO_DIR)
run("git pull --ff-only || true", allow_fail=True)

run("python3 -m pip install -q --upgrade pip setuptools wheel")
run("python3 -m pip install -q -r requirements-ci.txt")
run("python3 -m pip install -q \"numpy<2.1\" \"pydantic>=2.11,<3\" \"aiosqlite==0.20.0\" openai asyncpraw asyncprawcore python-dotenv PyYAML nest_asyncio google-api-python-client google-auth-oauthlib google-auth-httplib2 moviepy imageio-ffmpeg opencv-python-headless pillow scipy librosa soundfile psutil gputil pynvml \"yt-dlp>=2025.8.22\"")

qwen_cmd = "python3 -m pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git"
print(f"$ {qwen_cmd}")
qwen_res = subprocess.run(qwen_cmd, shell=True, text=True)
if qwen_res.returncode != 0:
    print("Warning: Qwen3-TTS install failed. Pipeline can still run with fallback TTS.")

print("Setup complete.")


In [ ]:
# Cell 2: Mount Drive + persist important directories to Drive
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path("/content/yotuubef")
os.chdir(REPO_DIR)

DRIVE_ROOT = Path("/content/drive/MyDrive/yotuubef")
PERSIST_ROOT = DRIVE_ROOT / "persistent"
RUNS_ROOT = DRIVE_ROOT / "runs"

important_dirs = {
    "backgrounds": PERSIST_ROOT / "backgrounds",
    "findings": PERSIST_ROOT / "findings",
    "processed": PERSIST_ROOT / "processed",
    "results": PERSIST_ROOT / "results",
    "logs": PERSIST_ROOT / "logs",
    "db": PERSIST_ROOT / "databases",
    "tokens": PERSIST_ROOT / "tokens",
}

for p in [DRIVE_ROOT, PERSIST_ROOT, RUNS_ROOT, *important_dirs.values()]:
    p.mkdir(parents=True, exist_ok=True)

def relink_dir(local_path: Path, drive_path: Path) -> None:
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.is_symlink():
        current_target = local_path.resolve(strict=False)
        if current_target == drive_path.resolve(strict=False):
            return
        local_path.unlink()
    elif local_path.exists():
        if not local_path.is_dir():
            raise RuntimeError(f"Cannot relink non-directory path: {local_path}")
        for item in local_path.iterdir():
            target_item = drive_path / item.name
            if target_item.exists():
                continue
            shutil.move(str(item), str(target_item))
        shutil.rmtree(local_path)

    local_path.symlink_to(drive_path, target_is_directory=True)

link_map = {
    REPO_DIR / "findings": important_dirs["findings"],
    REPO_DIR / "processed": important_dirs["processed"],
    REPO_DIR / "logs": important_dirs["logs"],
    REPO_DIR / "data" / "results": important_dirs["results"],
    REPO_DIR / "data" / "logs": important_dirs["logs"],
    REPO_DIR / "data" / "databases": important_dirs["db"],
}

for local_dir, drive_dir in link_map.items():
    relink_dir(local_dir, drive_dir)

print("Drive root:", DRIVE_ROOT)
print("Background folder:", important_dirs["backgrounds"])
print("Important outputs now persist to Google Drive automatically.")


In [ ]:
# Cell 3: Load Colab Secrets into env vars and enforce Drive background folder
import json
import os
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
DRIVE_ROOT = Path("/content/drive/MyDrive/yotuubef")
PERSIST_ROOT = DRIVE_ROOT / "persistent"
BACKGROUND_FOLDER_PATH = PERSIST_ROOT / "backgrounds"
TOKEN_FILE_PATH = PERSIST_ROOT / "tokens" / "youtube_token.json"

def get_secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value is not None else default
    except Exception:
        return os.getenv(name, default)

secret_to_env = {
    "REDDIT_CLIENT_ID": "REDDIT_CLIENT_ID",
    "REDDIT_CLIENT_SECRET": "REDDIT_CLIENT_SECRET",
    "REDDIT_USER_AGENT": "REDDIT_USER_AGENT",
    "NVIDIA_NIM_API_KEY": "NVIDIA_NIM_API_KEY",
    "NVIDIA_NIM_MODEL": "NVIDIA_NIM_MODEL",
    "NVIDIA_NIM_ALT_MODEL": "NVIDIA_NIM_ALT_MODEL",
    "NVIDIA_NIM_BASE_URL": "NVIDIA_NIM_BASE_URL",
    "GEMINI_API_KEY": "GEMINI_API_KEY",
    "BRAVE_SEARCH_API_KEY": "BRAVE_SEARCH_API_KEY",
    "HACKCLUB_SEARCH_API_KEY": "HACKCLUB_SEARCH_API_KEY",
}

for secret_name, env_name in secret_to_env.items():
    value = get_secret(secret_name, "")
    if value:
        os.environ[env_name] = value

os.environ.setdefault("REDDIT_USER_AGENT", "python:VideoBot:v2.0 (by /u/yourusername)")
os.environ["BACKGROUND_FOLDER"] = str(BACKGROUND_FOLDER_PATH)
os.environ["YOUTUBE_TOKEN_FILE"] = str(TOKEN_FILE_PATH)

youtube_token_json = get_secret("YOUTUBE_TOKEN_JSON", "")
if youtube_token_json:
    os.environ["YOUTUBE_TOKEN_JSON"] = youtube_token_json
    try:
        parsed = json.loads(youtube_token_json)
        TOKEN_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE_PATH.write_text(json.dumps(parsed), encoding="utf-8")
        print(f"Wrote YouTube token file to: {TOKEN_FILE_PATH}")
    except Exception as exc:
        print(f"Warning: Could not parse/write YOUTUBE_TOKEN_JSON ({exc})")

background_files = sorted(BACKGROUND_FOLDER_PATH.glob("*.mp4"))
print("BACKGROUND_FOLDER:", os.environ["BACKGROUND_FOLDER"])
print("Background videos found:", len(background_files))
if not background_files:
    print("Add .mp4 files to this Drive folder before running generation commands.")

required_core = ["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "NVIDIA_NIM_API_KEY"]
missing_core = [k for k in required_core if not os.environ.get(k)]
print("Missing required core secrets:", missing_core if missing_core else "none")
print("YouTube token available:", bool(os.environ.get("YOUTUBE_TOKEN_JSON") or TOKEN_FILE_PATH.exists()))


In [ ]:
# Cell 4: Runtime control panel (all CLI features)
COMMAND_MODE = "hybrid"  # find | single | batch | hybrid | manage | optimize | status | cleanup

# Common toggles used by 'find' and 'single'
DISABLE_CINEMATIC = False
DISABLE_AUDIO_DUCKING = False
DISABLE_AB_TESTING = False

# find mode
FIND_MAX_VIDEOS = 3
FIND_SUBREDDITS = ["speedrun", "HobbyDrama"]  # [] to use config defaults
FIND_SORT = "hot"  # hot | top | new | rising
FIND_TIME_FILTER = "day"  # hour | day | week | month
FIND_DRY_RUN = False

# single mode
SINGLE_REDDIT_URL = "https://www.reddit.com/r/speedrun/comments/abc123/example/"

# batch mode
BATCH_URLS = [
    "https://www.reddit.com/r/speedrun/comments/abc123/example/",
]
BATCH_MAX_CONCURRENT = 2

# hybrid mode
HYBRID_PROJECT_NAME = "colab_project_01"
HYBRID_REDDIT_URL = ""  # optional
HYBRID_RESUME = False
HYBRID_PHASE_OVERRIDE = ""  # IDEA_GENERATION | WAIT_FOR_GEMINI_REPORT | SYNTHESIS | EVIDENCE_GATHERING | SCRIPTING
HYBRID_GEMINI_REPORT_PATH = ""  # optional path to local report file
HYBRID_NO_UPLOAD = True
HYBRID_NO_AUTO_RESEARCH = False  # if True, pause for manual Gemini report instead of auto research

# manage mode
MANAGE_TIMEOUT_MINUTES = 20

# optimize mode
OPTIMIZE_FORCE = False

# cleanup mode
CLEANUP_LOGS = False
CLEANUP_ALL = False

# persistence
SYNC_SUMMARY_TO_DRIVE = True


In [ ]:
# Cell 5: Build and run command
import json
import os
import shlex
import subprocess
import threading
import time
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

REPO_DIR = Path("/content/yotuubef")
DRIVE_ROOT = Path("/content/drive/MyDrive/yotuubef")
RUNS_ROOT = DRIVE_ROOT / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def need_env(keys):
    missing = [k for k in keys if not os.environ.get(k)]
    if missing:
        raise ValueError(f"Missing required environment variables: {missing}")

def build_command():
    mode = COMMAND_MODE.strip().lower()
    cmd = ["python3", "main.py"]

    if mode == "find":
        need_env(["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "NVIDIA_NIM_API_KEY"])
        cmd += ["find", "--max-videos", str(FIND_MAX_VIDEOS), "--sort", FIND_SORT, "--time-filter", FIND_TIME_FILTER]
        if FIND_SUBREDDITS:
            cmd += ["--subreddits", *FIND_SUBREDDITS]
        if FIND_DRY_RUN:
            cmd.append("--dry-run")
        if DISABLE_CINEMATIC:
            cmd.append("--no-cinematic")
        if DISABLE_AUDIO_DUCKING:
            cmd.append("--no-audio-ducking")
        if DISABLE_AB_TESTING:
            cmd.append("--no-ab-testing")
        timeout = None

    elif mode == "single":
        need_env(["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "NVIDIA_NIM_API_KEY"])
        if not SINGLE_REDDIT_URL or "reddit.com" not in SINGLE_REDDIT_URL:
            raise ValueError("Set SINGLE_REDDIT_URL to a valid Reddit post URL.")
        cmd += ["single", SINGLE_REDDIT_URL]
        if DISABLE_CINEMATIC:
            cmd.append("--no-cinematic")
        if DISABLE_AUDIO_DUCKING:
            cmd.append("--no-audio-ducking")
        if DISABLE_AB_TESTING:
            cmd.append("--no-ab-testing")
        timeout = None

    elif mode == "batch":
        need_env(["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "NVIDIA_NIM_API_KEY"])
        if not BATCH_URLS:
            raise ValueError("BATCH_URLS is empty.")
        batch_file = REPO_DIR / "data" / "temp" / "colab_batch_urls.txt"
        batch_file.parent.mkdir(parents=True, exist_ok=True)
        batch_file.write_text("\n".join(BATCH_URLS) + "\n", encoding="utf-8")
        cmd += ["batch", str(batch_file), "--max-concurrent", str(BATCH_MAX_CONCURRENT)]
        timeout = None

    elif mode == "hybrid":
        need_env(["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "NVIDIA_NIM_API_KEY"])
        if not HYBRID_PROJECT_NAME.strip():
            raise ValueError("HYBRID_PROJECT_NAME cannot be empty.")
        cmd += ["hybrid", HYBRID_PROJECT_NAME]
        if HYBRID_REDDIT_URL.strip():
            cmd += ["--reddit-url", HYBRID_REDDIT_URL.strip()]
        if HYBRID_RESUME:
            cmd.append("--resume")
        if HYBRID_PHASE_OVERRIDE.strip():
            cmd += ["--phase", HYBRID_PHASE_OVERRIDE.strip()]
        if HYBRID_GEMINI_REPORT_PATH.strip():
            cmd += ["--gemini-report", HYBRID_GEMINI_REPORT_PATH.strip()]
        if HYBRID_NO_UPLOAD:
            cmd.append("--no-upload")
        if HYBRID_NO_AUTO_RESEARCH:
            cmd.append("--no-auto-research")
        timeout = None

    elif mode == "manage":
        cmd += ["manage"]
        timeout = max(1, int(MANAGE_TIMEOUT_MINUTES)) * 60

    elif mode == "optimize":
        cmd += ["optimize"]
        if OPTIMIZE_FORCE:
            cmd.append("--force")
        timeout = None

    elif mode == "status":
        cmd += ["status"]
        timeout = None

    elif mode == "cleanup":
        cmd += ["cleanup"]
        if CLEANUP_LOGS:
            cmd.append("--logs")
        if CLEANUP_ALL:
            cmd.append("--all")
        timeout = None

    else:
        raise ValueError(f"Unsupported COMMAND_MODE: {COMMAND_MODE}")

    return cmd, timeout

cmd, timeout_seconds = build_command()
cmd_str = " ".join(shlex.quote(part) for part in cmd)
print("Running command:")
print(cmd_str)

start = time.time()
timed_out = False
stdout_lines = []

def read_and_print(pipe):
    for line in pipe:
        print(line, end="", flush=True)
        stdout_lines.append(line)

proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
reader = threading.Thread(target=read_and_print, args=(proc.stdout,))
reader.daemon = True
reader.start()

try:
    proc.wait(timeout=timeout_seconds)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    timed_out = True
    stdout_lines.append(f"\nTimed out after {timeout_seconds} seconds.")
reader.join(timeout=2)

result = subprocess.CompletedProcess(
    cmd, proc.returncode,
    stdout="".join(stdout_lines),
    stderr="",
)

duration = round(time.time() - start, 2)

print(f"\nExit code: {result.returncode}")
print(f"Duration: {duration}s")
print("Timed out:", timed_out)

run_summary = {
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "command_mode": COMMAND_MODE,
    "command": cmd,
    "exit_code": result.returncode,
    "duration_seconds": duration,
    "timed_out": timed_out,
}

summary_path = REPO_DIR / "data" / "results" / "colab_last_run_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
print("Saved run summary to:", summary_path)

if SYNC_SUMMARY_TO_DRIVE:
    drive_summary = RUNS_ROOT / f"run_{int(time.time())}_summary.json"
    drive_summary.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
    print("Saved run summary to Drive:", drive_summary)

if result.returncode != 0:
    raise RuntimeError("Pipeline command failed. Review stdout/stderr above.")


In [ ]:
# Cell 6: Paste deep research report text and resume hybrid
import json
import shlex
import subprocess
import threading
import time
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")

HYBRID_PROJECT_NAME_RESUME = (
    HYBRID_PROJECT_NAME if "HYBRID_PROJECT_NAME" in globals() else "colab_project_01"
)
HYBRID_PASTED_REPORT_TEXT = """
PASTE_YOUR_DEEP_RESEARCH_REPORT_HERE
"""
HYBRID_RESUME_NO_UPLOAD = True

def find_latest_hybrid_result(results_dir: Path):
    files = sorted(
        results_dir.glob("results_hybrid_*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return files[0] if files else None

report_text = HYBRID_PASTED_REPORT_TEXT.strip()
if not report_text:
    raise ValueError("HYBRID_PASTED_REPORT_TEXT is empty. Paste your deep research output first.")
if report_text == "PASTE_YOUR_DEEP_RESEARCH_REPORT_HERE":
    raise ValueError("HYBRID_PASTED_REPORT_TEXT still has the placeholder text. Replace it with your deep research output.")

project_dir = REPO_DIR / "findings" / HYBRID_PROJECT_NAME_RESUME
report_dir = project_dir / "research" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "gemini_report_colab_pasted.txt"
report_path.write_text(report_text, encoding="utf-8")
print("Saved pasted report to:", report_path)

cmd = [
    "python3",
    "main.py",
    "hybrid",
    HYBRID_PROJECT_NAME_RESUME,
    "--resume",
    "--gemini-report",
    str(report_path),
]
if HYBRID_RESUME_NO_UPLOAD:
    cmd.append("--no-upload")

cmd_str = " ".join(shlex.quote(part) for part in cmd)
print("Running command:")
print(cmd_str)

start = time.time()
stdout_lines = []

def read_and_print(pipe):
    for line in pipe:
        print(line, end="", flush=True)
        stdout_lines.append(line)

proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
reader = threading.Thread(target=read_and_print, args=(proc.stdout,))
reader.daemon = True
reader.start()
proc.wait()
reader.join(timeout=2)

duration = round(time.time() - start, 2)
print(f"\nExit code: {proc.returncode}")
print(f"Duration: {duration}s")

state_path = project_dir / "research" / "state.json"
if state_path.exists():
    try:
        state_payload = json.loads(state_path.read_text(encoding="utf-8"))
        print("State status:", state_payload.get("status", "unknown"))
        print("Current phase:", state_payload.get("current_phase", "unknown"))
    except Exception as exc:
        print(f"Warning: failed to read state.json ({exc})")

latest_result = find_latest_hybrid_result(REPO_DIR / "data" / "results")
if latest_result and latest_result.exists():
    try:
        payload = json.loads(latest_result.read_text(encoding="utf-8"))
        print("Latest hybrid result:", latest_result)
        print("Result status:", payload.get("status", "unknown"))
        print("Result phase:", payload.get("current_phase", "unknown"))
        if payload.get("final_script_path"):
            print("Final script:", payload.get("final_script_path"))
        if payload.get("workspace_path"):
            print("Workspace:", payload.get("workspace_path"))
    except Exception as exc:
        print(f"Warning: failed to parse latest hybrid result ({exc})")

if proc.returncode != 0:
    raise RuntimeError("Hybrid resume failed. Review output above.")


In [ ]:
# Cell 6: Save a lightweight run snapshot manifest to Drive
import json
import shutil
import time
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
DRIVE_ROOT = Path("/content/drive/MyDrive/yotuubef")
RUNS_ROOT = DRIVE_ROOT / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

snapshot_dir = RUNS_ROOT / f"snapshot_{int(time.time())}"
snapshot_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "paths": {},
}

def capture_file(src: Path, name: str) -> None:
    if not src.exists() or not src.is_file():
        return
    dst = snapshot_dir / name
    shutil.copy2(src, dst)
    manifest["paths"][name] = str(dst)

capture_file(REPO_DIR / "data" / "results" / "colab_last_run_summary.json", "colab_last_run_summary.json")
capture_file(REPO_DIR / "run_log.txt", "run_log.txt")

result_files = sorted((REPO_DIR / "data" / "results").glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if result_files:
    capture_file(result_files[0], "latest_pipeline_result.json")

state_files = sorted((REPO_DIR / "findings").glob("*/research/state.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if state_files:
    capture_file(state_files[0], "latest_findings_state.json")

manifest_path = snapshot_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Snapshot saved to:", snapshot_dir)
print("Manifest:", manifest_path)


In [ ]:
# Cell 7: Inspect latest outputs and Drive-backed folders
import os
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
background_dir = Path(os.environ.get("BACKGROUND_FOLDER", ""))
results_dir = REPO_DIR / "data" / "results"
processed_dir = REPO_DIR / "processed"
findings_dir = REPO_DIR / "findings"

print("BACKGROUND_FOLDER:", background_dir)
if background_dir.exists():
    bg_videos = sorted(background_dir.glob("*.mp4"))
    print("Background .mp4 count:", len(bg_videos))
else:
    print("Background folder not found.")

print("\nLatest result files:")
if results_dir.exists():
    for p in sorted(results_dir.glob("*.json"), key=lambda x: x.stat().st_mtime, reverse=True)[:5]:
        print(" -", p)
else:
    print("No results directory yet.")

print("\nLatest processed videos:")
if processed_dir.exists():
    for p in sorted(processed_dir.glob("*.mp4"), key=lambda x: x.stat().st_mtime, reverse=True)[:5]:
        print(" -", p)
else:
    print("No processed directory yet.")

print("\nFindings projects:")
if findings_dir.exists():
    projects = sorted([p for p in findings_dir.iterdir() if p.is_dir()], key=lambda x: x.stat().st_mtime, reverse=True)
    if projects:
        for p in projects[:10]:
            print(" -", p.name)
    else:
        print("No findings projects yet.")
else:
    print("No findings directory yet.")


In [ ]:
# Cell 8: Optional download of latest processed video
from pathlib import Path
from google.colab import files

processed_dir = Path("/content/yotuubef/processed")
videos = sorted(processed_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True) if processed_dir.exists() else []

if videos:
    latest = videos[0]
    print("Downloading:", latest)
    files.download(str(latest))
else:
    print("No processed .mp4 files found.")
